# FINAL 02 — Leakage-Safe Model Development, Feature Selection and Configuration Lock

**Thesis:** An Ethical Risk Assessment of Sentiment Analysis Models in Social Media Monitoring with a Focus on Bias, Inclusivity, and Misclassification Patterns  
**Student:** Abhishek Tomar | PN1196973 | LJMU | October 2026

**Purpose:** Produce the five-model common-representation comparison and select/tune the final pipeline using only TweetEval train/validation/CV evidence before one held-out test evaluation.

> **Final-run rule:** TweetEval test data is held out from model/feature/hyperparameter selection. External datasets are never used for source-model selection. Linguistic subgroup thresholds are frozen and are not tuned after observing subgroup sizes.

In [2]:
!pip install -q emoji vaderSentiment pandas pyarrow scikit-learn scipy google-api-python-client

from google.colab import drive, auth
drive.mount('/content/drive')

import io
import sys
from pathlib import Path
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Shared final-run utility stored in the Drive Experiments folder.
UTIL_FILE_ID = "14KfwtUf2zSd_jlW7i-6K8XgBJ62Aqk02"
UTIL_LOCAL = Path("/content/final_thesis_utils.py")

def load_final_thesis_utils():
    if not UTIL_LOCAL.exists() or UTIL_LOCAL.stat().st_size == 0:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        service = build("drive", "v3", credentials=creds, cache_discovery=False)
        request = service.files().get_media(fileId=UTIL_FILE_ID)
        with io.FileIO(str(UTIL_LOCAL), "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
                if status:
                    print(f"Downloading final_thesis_utils.py: {int(status.progress()*100)}%")

    assert UTIL_LOCAL.exists() and UTIL_LOCAL.stat().st_size > 0, \
        "final_thesis_utils.py could not be downloaded."

    if "/content" not in sys.path:
        sys.path.insert(0, "/content")

    import importlib
    import final_thesis_utils as ftu
    ftu = importlib.reload(ftu)

    RUN_ROOT = Path("/content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN")
    ftu.FINAL_ROOT = RUN_ROOT
    ftu.FINAL = {
        "root": RUN_ROOT,
        "data": RUN_ROOT / "data",
        "models": RUN_ROOT / "models",
        "predictions": RUN_ROOT / "predictions",
        "results": RUN_ROOT / "results",
        "figures": RUN_ROOT / "figures",
        "manifests": RUN_ROOT / "manifests",
    }
    return ftu

ftu = load_final_thesis_utils()

globals().update({k: v for k, v in vars(ftu).items() if not k.startswith("_")})
FINAL_ROOT = ftu.FINAL_ROOT
FINAL = ftu.FINAL
setup_final_dirs()

print("✅ final_thesis_utils imported successfully")
print("✅ Final run root:", FINAL_ROOT)

!pip install -q xgboost
import json, time, pickle
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier


Mounted at /content/drive


✅ final_thesis_utils imported successfully
✅ Final run root: /content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN


In [3]:
tr=pd.read_parquet(FINAL["data"]/"tweeteval_train_final.parquet")
va=pd.read_parquet(FINAL["data"]/"tweeteval_validation_final.parquet")
te=pd.read_parquet(FINAL["data"]/"tweeteval_test_final.parquet")

Xtr,ytr=tr.text.astype(str),tr.label_id.to_numpy()
Xva,yva=va.text.astype(str),va.label_id.to_numpy()
Xte,yte=te.text.astype(str),te.label_id.to_numpy()

word_vec=lambda: TfidfVectorizer(preprocessor=clean_text,ngram_range=(1,2),min_df=2,max_df=0.98,sublinear_tf=True,max_features=120000)
char_vec=lambda: TfidfVectorizer(preprocessor=clean_text,analyzer="char_wb",ngram_range=(3,5),min_df=2,sublinear_tf=True,max_features=120000)

def baseline_models():
    return {
        "LogisticRegression":LogisticRegression(C=1.0,max_iter=2500,solver="liblinear",random_state=SEED),
        "LinearSVM":CalibratedClassifierCV(LinearSVC(C=1.0,random_state=SEED),cv=3),
        "MultinomialNB":MultinomialNB(alpha=0.1),
        "RandomForest":RandomForestClassifier(n_estimators=300,min_samples_leaf=1,n_jobs=-1,random_state=SEED,class_weight="balanced_subsample"),
        "XGBoost":XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.15,subsample=0.9,colsample_bytree=0.8,
                                objective="multi:softprob",num_class=3,eval_metric="mlogloss",n_jobs=-1,random_state=SEED),
    }

## Stage A — Common-representation five-model comparison on validation only
The held-out TweetEval test set is not evaluated in this stage.

In [4]:
baseline_rows=[]
baseline_pipes={}
for name,clf in baseline_models().items():
    print("Fitting",name)
    pipe=Pipeline([("features",word_vec()),("clf",clf)])
    pipe.fit(Xtr,ytr)
    pred=pipe.predict(Xva)
    proba=pipe.predict_proba(Xva)
    row=evaluate_predictions(yva,pred,proba,name)
    baseline_rows.append(row)
    baseline_pipes[name]=pipe

baseline_val=pd.DataFrame(baseline_rows).sort_values("Macro F1",ascending=False)
baseline_val.to_csv(FINAL["results"]/"F02_baseline_validation.csv",index=False)
display(baseline_val)
BEST_BASELINE=baseline_val.iloc[0]["Model"]
print("Validation-selected baseline:",BEST_BASELINE)

Fitting LogisticRegression
Fitting LinearSVM
Fitting MultinomialNB
Fitting RandomForest
Fitting XGBoost


,Model,Accuracy,Macro F1,Weighted F1,Macro Precision,Macro Recall,MCC,Cohen Kappa,Brier
1,LinearSVM,0.6860,0.646191,0.681900,0.674180,0.632375,0.484520,0.481360,0.146418
0,LogisticRegression,0.6705,0.609972,0.659555,0.673909,0.594064,0.454766,0.446748,0.150729
4,XGBoost,0.6560,0.596196,0.645258,0.659035,0.581605,0.436952,0.422749,0.153532
2,MultinomialNB,0.6260,0.577419,0.618495,0.615745,0.563893,0.380105,0.377377,0.161933
3,RandomForest,0.6200,0.511543,0.590066,0.669806,0.514111,0.372255,0.346319,0.165758


Validation-selected baseline: LinearSVM


## Stage B — Feature representation comparison using the validation-selected model family
Selection remains validation-only.

In [5]:
def fresh_clf(name):
    return baseline_models()[name]

feature_sets={
    "word_tfidf": word_vec(),
    "char_ngrams": char_vec(),
    "social_only": SocialFeatureTransformer(),
    "hybrid": FeatureUnion([
        ("word",word_vec()),
        ("char",char_vec()),
        ("social",SocialFeatureTransformer()),
    ])
}
feature_rows=[]
for fname,features in feature_sets.items():
    print("Feature set",fname)
    pipe=Pipeline([("features",features),("clf",fresh_clf(BEST_BASELINE))])
    pipe.fit(Xtr,ytr)
    pred=pipe.predict(Xva); proba=pipe.predict_proba(Xva)
    row=evaluate_predictions(yva,pred,proba,f"{BEST_BASELINE}:{fname}")
    row["Feature Set"]=fname
    feature_rows.append(row)

feature_val=pd.DataFrame(feature_rows).sort_values("Macro F1",ascending=False)
feature_val.to_csv(FINAL["results"]/"F03_feature_validation.csv",index=False)
display(feature_val)
BEST_FEATURE=feature_val.iloc[0]["Feature Set"]
print("Validation-selected feature set:",BEST_FEATURE)

Feature set word_tfidf
Feature set char_ngrams
Feature set social_only
Feature set hybrid


,Model,Accuracy,Macro F1,Weighted F1,Macro Precision,Macro Recall,MCC,Cohen Kappa,Brier,Feature Set
3,LinearSVM:hybrid,0.714,0.680448,0.710506,0.710569,0.664690,0.530664,0.527883,0.137613,hybrid
0,LinearSVM:word_tfidf,0.686,0.646191,0.681900,0.674180,0.632375,0.484520,0.481360,0.146418,word_tfidf
1,LinearSVM:char_ngrams,0.687,0.644051,0.681563,0.681839,0.628161,0.485422,0.480603,0.144009,char_ngrams
2,LinearSVM:social_only,0.579,0.468370,0.550887,0.550953,0.476810,0.287809,0.279873,0.177821,social_only


Validation-selected feature set: hybrid


## Stage C — Hyperparameter tuning of the selected model/feature configuration
Grid search uses **training data only**. The validation set confirms the selected configuration. No test score is available yet.

In [6]:
def selected_features(name):
    return {
        "word_tfidf":word_vec(),
        "char_ngrams":char_vec(),
        "social_only":SocialFeatureTransformer(),
        "hybrid":FeatureUnion([("word",word_vec()),("char",char_vec()),("social",SocialFeatureTransformer())])
    }[name]

base=fresh_clf(BEST_BASELINE)
pipe=Pipeline([("features",selected_features(BEST_FEATURE)),("clf",base)])

if BEST_BASELINE=="LogisticRegression":
    grid={"clf__C":[0.5,1,2,5,10]}
elif BEST_BASELINE=="MultinomialNB":
    grid={"clf__alpha":[0.05,0.1,0.5,1.0]}
elif BEST_BASELINE=="RandomForest":
    grid={"clf__n_estimators":[200,400],"clf__min_samples_leaf":[1,2]}
elif BEST_BASELINE=="XGBoost":
    grid={"clf__max_depth":[4,6],"clf__learning_rate":[0.1,0.2],"clf__n_estimators":[250,400]}
else:
    # Calibrated LinearSVC: estimator parameter path in current sklearn
    grid={"clf__estimator__C":[0.5,1,2]}

gs=GridSearchCV(pipe,grid,scoring="f1_macro",cv=5,n_jobs=-1,refit=True)
gs.fit(Xtr,ytr)
val_pred=gs.predict(Xva); val_proba=gs.predict_proba(Xva)
tune_val=evaluate_predictions(yva,val_pred,val_proba,f"{BEST_BASELINE}:{BEST_FEATURE}:tuned")
tune_val["Best Params"]=json.dumps(gs.best_params_)
tune_val["CV Best Macro F1"]=gs.best_score_
pd.DataFrame([tune_val]).to_csv(FINAL["results"]/"F04_tuning_validation.csv",index=False)
print(tune_val)

{'Model': 'LinearSVM:hybrid:tuned', 'Accuracy': 0.723, 'Macro F1': 0.689215112918855, 'Weighted F1': 0.7198128545535926, 'Macro Precision': 0.7155514441963509, 'Macro Recall': 0.6746142980320196, 'MCC': np.float64(0.5459444780526475), 'Cohen Kappa': np.float64(0.5436163543514203), 'Brier': 0.13515368297663108, 'Best Params': '{"clf__estimator__C": 0.5}', 'CV Best Macro F1': np.float64(0.6508645573800363)}


## Stage D — Lock, refit on train+validation, and evaluate TweetEval test once
This is the first point at which the TweetEval test labels are used for model evaluation.

In [7]:
lock={
    "best_baseline":BEST_BASELINE,
    "best_feature_set":BEST_FEATURE,
    "best_params":gs.best_params_,
    "selection_evidence":"TweetEval train + validation/CV only",
    "test_used_for_selection":False,
    "external_data_used_for_selection":False,
}
save_json(lock,FINAL["manifests"]/"F04_FINAL_CONFIGURATION_LOCK.json")
print(json.dumps(lock,indent=2))

trainval=pd.concat([tr,va],ignore_index=True)
Xtv=trainval.text.astype(str); ytv=trainval.label_id.to_numpy()
final_pipe=gs.best_estimator_
final_pipe.fit(Xtv,ytv)
test_pred=final_pipe.predict(Xte); test_proba=final_pipe.predict_proba(Xte)
final_test=evaluate_predictions(yte,test_pred,test_proba,"FinalSelectedPipeline")
pd.DataFrame([final_test]).to_csv(FINAL["results"]/"F04_final_tweeteval_test.csv",index=False)
save_pickle(final_pipe,FINAL["models"]/"FINAL_SELECTED_PIPELINE.pkl")
pf=prediction_frame(te,yte,test_pred,test_proba,"FinalSelectedPipeline","TweetEval")
pf.to_parquet(FINAL["predictions"]/"FINAL_SELECTED_TweetEval.parquet",index=False)
print(final_test)

{
  "best_baseline": "LinearSVM",
  "best_feature_set": "hybrid",
  "best_params": {
    "clf__estimator__C": 0.5
  },
  "selection_evidence": "TweetEval train + validation/CV only",
  "test_used_for_selection": false,
  "external_data_used_for_selection": false
}
{'Model': 'FinalSelectedPipeline', 'Accuracy': 0.6213774014978835, 'Macro F1': 0.6053146348313904, 'Weighted F1': 0.61744280593564, 'Macro Precision': 0.6221513624746513, 'Macro Recall': 0.6125385459713701, 'MCC': np.float64(0.3965103610029489), 'Cohen Kappa': np.float64(0.39038531561119005), 'Brier': 0.16523882943465976}


## Stage E — Lock five common-FS1 model-family pipelines for RQ2
These are refitted on train+validation using the predeclared common representation, then evaluated once on the same held-out test partition. They are **not ranked as an overall ethical choice**.

In [8]:
model_test_rows=[]
for name,clf in baseline_models().items():
    pipe=Pipeline([("features",word_vec()),("clf",clf)])
    pipe.fit(Xtv,ytv)
    yp=pipe.predict(Xte); pp=pipe.predict_proba(Xte)
    row=evaluate_predictions(yte,yp,pp,name); model_test_rows.append(row)
    save_pickle(pipe,FINAL["models"]/f"RQ2_{name}_FS1.pkl")
    prediction_frame(te,yte,yp,pp,name,"TweetEval").to_parquet(
        FINAL["predictions"]/f"RQ2_{name}_TweetEval.parquet",index=False
    )
model_test=pd.DataFrame(model_test_rows)
model_test.to_csv(FINAL["results"]/"F02_five_model_test_performance.csv",index=False)
display(model_test)

,Model,Accuracy,Macro F1,Weighted F1,Macro Precision,Macro Recall,MCC,Cohen Kappa,Brier
0,LogisticRegression,0.582791,0.540018,0.557249,0.602403,0.552663,0.326131,0.305901,0.176890
1,LinearSVM,0.591908,0.570904,0.585367,0.591937,0.576612,0.345232,0.338328,0.175458
2,MultinomialNB,0.566428,0.542520,0.558074,0.565440,0.547009,0.300211,0.293214,0.183072
3,RandomForest,0.543878,0.427984,0.456406,0.632860,0.470751,0.251251,0.190310,0.195116
4,XGBoost,0.566835,0.481532,0.509097,0.633359,0.499862,0.291788,0.242527,0.191859
